In [0]:
%pip install "shapely>=2.1" pyproj

In [0]:
from pyspark.sql import functions as F, Window
from dq import DQLog  # dq.py at the repo root
from tiers import load_assumptions, load_paces, reach_m, tier_expr  # tiers.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("source_schema", "silver")  # dev_<name> for test runs
dbutils.widgets.text("target_schema", "gold")    # dev_<name>_gold for test runs
SRC= f"{CATALOG}.{dbutils.widgets.get('source_schema')}"
OUT= f"{CATALOG}.{dbutils.widgets.get('target_schema')}"
assert SRC != OUT, "source and target must differ: silver and gold both have a centres table"
print("reading", SRC, "| writing to", OUT)
dq= DQLog(spark, "04_gold_access", schema=f"{CATALOG}.silver" if OUT.endswith(".gold") else OUT)  # dev runs log to the dev schema

A= load_assumptions()
PACES= load_paces(A)                       # walk.pace.senior / typical / brisk
LIMIT_MIN= A["walk.limit_min"]["value"]    # 15
CUTOFF_M= A["walk.cutoff_m"]["value"]      # 2000, as in silver.dist_matrix
SENS_LIMITS= sorted({LIMIT_MIN, *(int(x) for x in str(A["access.sensitivity_limits_min"]["value"]).split(","))})
print("paces", PACES, "| limit", LIMIT_MIN, "min =", [round(r) for r in reach_m(LIMIT_MIN, PACES)], "m | sensitivity", SENS_LIMITS)

In [0]:
# nearest, second and third centre per block; ties broken by centre_id, the same rule as gold.block_options (#6)
dm= spark.table(f"{SRC}.dist_matrix").filter(F.col("walk_m") <= CUTOFF_M)
w= Window.partitionBy("block_id").orderBy("walk_m", "centre_id")
ranked= dm.withColumn("rank", F.row_number().over(w)).filter("rank <= 3")
wide= (ranked.groupBy("block_id")
       .pivot("rank", [1, 2, 3]).agg(F.first("centre_id").alias("centre_id"), F.first("walk_m").alias("walk_m")))
for r, name in [(1, "nearest"), (2, "second"), (3, "third")]:
    wide= wide.withColumnRenamed(f"{r}_centre_id", f"{name}_centre_id").withColumnRenamed(f"{r}_walk_m", f"{name}_walk_m")

blocks= spark.table(f"{SRC}.blocks")
access= (blocks
    .select("block_id", F.concat_ws(" ", "blk_no", "street").alias("block_label"), F.col("subzone_c").alias("subzone_id"),
            "pln_area_n", "lon", "lat", "est_seniors")
    .join(wide, "block_id", "left")  # a block with no centre within 2 km keeps nulls, and is T4
    .withColumn("tier_now", tier_expr(F.col("nearest_walk_m"), LIMIT_MIN, PACES))
    .withColumn("tier_if_nearest_closes", tier_expr(F.col("second_walk_m"), LIMIT_MIN, PACES))
    .select("block_id", "block_label", "subzone_id", "pln_area_n", "lon", "lat", "est_seniors",
            "nearest_centre_id", "nearest_walk_m", "tier_now",
            "second_centre_id", "second_walk_m", "tier_if_nearest_closes",
            "third_centre_id", "third_walk_m"))
access.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.block_access")

a= spark.table(f"{OUT}.block_access")
n_blocks= blocks.count()
dq.step("block_access", n_blocks, a.filter(F.col("nearest_centre_id").isNotNull()).count(), "blocks with a centre within 2 km")
display(a.groupBy("tier_now").agg(F.count("*").alias("blocks"), F.round(F.sum("est_seniors")).alias("seniors")).orderBy("tier_now"))

In [0]:
# checks: one row per block, tiers agree with the distances, and ranks 1-3 match Spec 2's gold.block_options
assert a.count() == n_blocks and a.select("block_id").distinct().count() == n_blocks, "block_access is not one row per block"
assert a.filter(F.col("subzone_id").isNull()).count() == 0, "a block has no subzone_id"
assert a.filter(F.col("nearest_centre_id").isNull() & (F.col("tier_now") != "T4")).count() == 0, "a block with no centre is not T4"
assert a.filter(F.col("second_walk_m") < F.col("nearest_walk_m")).count() == 0, "second centre is nearer than the nearest"
assert a.filter(F.col("third_walk_m") < F.col("second_walk_m")).count() == 0, "third centre is nearer than the second"

# logged, not asserted: in the job 05_gold_closures may not have rebuilt block_options yet
if not spark.catalog.tableExists(f"{OUT}.block_options"):
    print(f"skipped: {OUT}.block_options doesn't exist yet")
else:
    opts= spark.table(f"{OUT}.block_options").filter("rank <= 3").select("block_id", "rank", F.col("centre_id").alias("opt_centre_id"))
    mine= a.select("block_id", F.posexplode(F.array("nearest_centre_id", "second_centre_id", "third_centre_id")).alias("pos", "centre_id"))
    mism= (mine.filter(F.col("centre_id").isNotNull()).withColumn("rank", F.col("pos") + 1)
           .join(opts, ["block_id", "rank"], "full_outer").filter(~F.col("centre_id").eqNullSafe(F.col("opt_centre_id"))))
    n_mism= mism.count()
    dq.issues("block_access_vs_block_options", [(r["block_id"], f"rank {r['rank']}: {r['centre_id']} vs {r['opt_centre_id']}") for r in mism.limit(200).collect()])
    print("ranks 1-3 match block_options" if n_mism == 0 else f"WARNING: {n_mism} block/rank pairs disagree with block_options")

In [0]:
# single-closure outcomes for every operating centre at each time limit (ASSUMPTIONS.md, Outcome groups)
centres= spark.table(f"{SRC}.centres")
n_centres= centres.count()
closable= centres.filter(~F.col("is_interim"))  # interim centres take diners but can't be picked to close
n_closable= closable.count()
moves= a.filter(F.col("nearest_centre_id").isNotNull()).select(
    F.col("nearest_centre_id").alias("closed_centre_id"), "block_id", "est_seniors", "nearest_walk_m", "second_walk_m")
parts= []
for limit in SENS_LIMITS:
    before, after= tier_expr(F.col("nearest_walk_m"), limit, PACES), tier_expr(F.col("second_walk_m"), limit, PACES)
    cut= (before == "T1") & (after != "T1")
    parts.append(moves.groupBy("closed_centre_id").agg(
        F.sum(F.when(cut, F.col("est_seniors")).otherwise(0)).alias("seniors_cut_off"),
        F.sum(F.when((before == "T1") & (after == "T4"), F.col("est_seniors")).otherwise(0)).alias("seniors_no_one_can_walk"),
        F.sum(F.when(before != "T1", F.col("est_seniors")).otherwise(0)).alias("seniors_already_without"),
        F.sum(F.when(before.isin("T2", "T3") & (after == "T4"), F.col("est_seniors")).otherwise(0)).alias("seniors_now_worse"),
        F.sum(F.when(cut, 1).otherwise(0)).alias("blocks_cut_off"))
        .withColumn("limit_min", F.lit(limit)))
counts= ["seniors_cut_off", "seniors_no_one_can_walk", "seniors_already_without", "seniors_now_worse", "blocks_cut_off"]
by_limit= parts[0]
for p in parts[1:]:
    by_limit= by_limit.unionByName(p)
grid= closable.select(F.col("centre_id").alias("closed_centre_id")).crossJoin(spark.createDataFrame([(l,) for l in SENS_LIMITS], "limit_min int"))
sens= (grid.join(by_limit, ["closed_centre_id", "limit_min"], "left")  # a centre that is no block's nearest closes with no one affected
       .fillna(0, subset=counts)
       .select("closed_centre_id", "limit_min", *counts))
sens.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.access_sensitivity")

s= spark.table(f"{OUT}.access_sensitivity")
assert s.count() == n_closable * len(SENS_LIMITS), "access_sensitivity is not one row per closable centre per limit"
assert s.filter(F.col("seniors_no_one_can_walk") > F.col("seniors_cut_off") + 1e-9).count() == 0, "no-one-can-walk is a sub-line of cut off"
dq.step("access_sensitivity", n_closable * len(SENS_LIMITS), s.count(), "one row per closable (non-interim) centre per time limit")
display(s.filter(F.col("limit_min") == LIMIT_MIN).orderBy(F.desc("seniors_cut_off")).limit(10))

In [0]:
# shared centre and closure tables for Pages 1 and 2 and Spec 2
(centres.select("centre_id", "name", "lon", "lat", "cooked_food_stalls", "is_interim", "pln_area_n", F.col("subzone_c").alias("subzone_id"))
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.centres"))
closures= spark.table(f"{SRC}.closures")
(closures.select("closure_id", "centre_id", "type", "start_date", "end_date", "duration_days", "remarks")
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.centre_closures"))

# a centre no HDB block can reach within 2 km is in gold.centres but drops out of the access model
reached= dm.select("centre_id").distinct()
excluded= centres.join(reached, "centre_id", "left_anti").select("centre_id", "name").collect()
dq.issues("centre_excluded", [(r["centre_id"], f"{r['name']}: no HDB block within {CUTOFF_M} m on the walking network") for r in excluded])
print(f"operating centres {n_centres} = in the access model {n_centres - len(excluded)} + excluded {len(excluded)}")
dq.step("centres_in_access_model", n_centres, n_centres - len(excluded), "operating centres reachable from an HDB block")
dq.step("centre_closures", closures.count(), spark.table(f"{OUT}.centre_closures").count(), "every closure type and date")

In [0]:
# subzone senior shares from measured SingStat HDB totals (silver.subzone_population), not the block estimates
SUPER_AGED= A["access.super_aged_share"]["value"]        # 0.21
MIN_RES= A["access.min_subzone_residents"]["value"]      # 50

sz= spark.table(f"{SRC}.subzones").select(F.col("subzone_c").alias("subzone_id"), F.col("subzone_n").alias("subzone_name"), "pln_area_n")
pop= spark.table(f"{SRC}.subzone_population").select(F.col("subzone_c").alias("subzone_id"), "hdb_residents", "hdb_seniors")
hdb= pop.filter(F.col("hdb_residents") > 0)  # SingStat lists every subzone; most non-HDB ones are 0
# national share over every matched subzone; residents under unmatched SingStat names are logged by 02_silver
nat= pop.agg((F.sum("hdb_seniors") / F.sum("hdb_residents")).alias("p")).first()["p"]
print(f"national HDB share aged 65+: {nat:.4f}")

big_enough= F.col("hdb_residents") >= MIN_RES
pct= F.when(big_enough, F.col("hdb_seniors") / F.col("hdb_residents"))
seniors= (sz.join(hdb, "subzone_id", "inner")  # subzones with no HDB residents get no row
    .withColumn("pct_65p", pct)
    .withColumn("multiplier", F.col("pct_65p") / F.lit(nat))
    .withColumn("is_super_aged", F.col("pct_65p") >= SUPER_AGED)  # null where the subzone is too small to say
    .select("subzone_id", "subzone_name", "pln_area_n", "hdb_residents", "hdb_seniors", "pct_65p", "multiplier", "is_super_aged"))
seniors.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.subzone_seniors")

ss= spark.table(f"{OUT}.subzone_seniors")
small= ss.filter(~big_enough).collect()
dq.issues("subzone_too_small", [(r["subzone_id"], f"{r['subzone_name']}: {r['hdb_residents']} HDB residents, under {MIN_RES}; no share or multiplier") for r in small])
dq.step("subzone_seniors", pop.count(), ss.count(), "SingStat subzones kept: those with HDB residents")
print("subzones:", ss.count(), "| too small:", len(small), "| super-aged:", ss.filter("is_super_aged").count())
display(ss.orderBy(F.desc("pct_65p")).limit(10))

In [0]:
# light subzone outlines for the app: repair, simplify to ~15 m in SVY21 metres, back to lon/lat
import json
import shapely
from pyproj import Transformer
from shapely.geometry import shape, mapping
from shapely.ops import transform

SIMPLIFY_M= 15
to_svy21= Transformer.from_crs("EPSG:4326", "EPSG:3414", always_xy=True).transform
to_wgs84= Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True).transform

def polygonal(g):
    # make_valid can return a collection with stray lines or points; keep only the area
    parts= [p for p in shapely.get_parts(g) if p.geom_type in ("Polygon", "MultiPolygon")]
    return shapely.union_all(parts) if parts else shapely.Polygon()

raw= spark.table(f"{SRC}.subzones").select("subzone_c", "subzone_n", "pln_area_n", "geometry").toPandas()
svy= [transform(to_svy21, polygonal(shapely.make_valid(shape(json.loads(g))))) for g in raw["geometry"]]

# simplify the subzones as one coverage, so neighbours keep a shared border with no gaps or overlaps
simple= list(shapely.coverage_simplify(svy, SIMPLIFY_M))
if not all(g.is_valid and not g.is_empty for g in simple):
    print("coverage simplify gave invalid shapes; simplifying each subzone on its own")
    simple= [g.simplify(SIMPLIFY_M, preserve_topology=True) for g in svy]
geoms= [polygonal(transform(to_wgs84, g)) for g in simple]

bad= [c for c, g in zip(raw["subzone_c"], geoms) if not g.is_valid or g.is_empty]
assert not bad, f"invalid or empty shapes after simplifying: {bad}"
shapes= raw[["subzone_c", "subzone_n", "pln_area_n"]].rename(columns={"subzone_c": "subzone_id", "subzone_n": "name", "pln_area_n": "planning_area"})
shapes["geometry"]= [json.dumps(mapping(g)) for g in geoms]

# every subzone with seniors needs an outline to draw; checked before writing
no_shape= spark.table(f"{OUT}.subzone_seniors").filter(~F.col("subzone_id").isin(list(shapes["subzone_id"]))).select("subzone_id", "subzone_name").collect()
dq.issues("subzone_no_shape", [(r["subzone_id"], r["subzone_name"]) for r in no_shape])
assert not no_shape, f"{len(no_shape)} subzones in subzone_seniors have no shape"

(spark.createDataFrame(shapes, schema="subzone_id string, name string, planning_area string, geometry string")
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.subzone_shapes"))

before= raw["geometry"].str.len().sum()
after= shapes["geometry"].str.len().sum()
print(f"subzone_shapes: {len(shapes)} shapes | GeoJSON {before / 1e6:.1f} MB -> {after / 1e6:.2f} MB")
dq.step("subzone_shapes", len(raw), len(shapes), f"simplified to {SIMPLIFY_M} m; GeoJSON {after / 1e6:.2f} MB")